# Garden Store Agent

A conversational agent that uses the Garden Store A2A interface via a local Ollama model.

**Before running:**
- Garden store running: `docker compose up`
- Ollama running with a model pulled: `ollama pull qwen2.5:7b`

Run cells 1–3 once to initialise, then use the chat widget in cell 4.

In [ ]:
import json

import httpx
import ipywidgets as widgets
from IPython.display import display
from openai import OpenAI

# ---------------------------------------------------------------------------
# Configuration — adjust if Ollama or the store are on a different host
# ---------------------------------------------------------------------------
OLLAMA_BASE  = "http://localhost:11434"  # Windows host from WSL: use host IP
OLLAMA_MODEL = "qwen2.5:7b"
GARDEN_STORE = "http://localhost:8000"

In [ ]:

# ---------------------------------------------------------------------------
# A2A helpers
# ---------------------------------------------------------------------------

def fetch_card(store_base: str) -> dict:
    r = httpx.get(f"{store_base}/.well-known/agent-card.json", timeout=10)
    r.raise_for_status()
    return r.json()


def build_system_prompt(card: dict) -> str:
    skills = card.get("skills", [])
    skills_text = "\n\n".join(
        "Action: {name}\n{description}\nExample intent: {example}".format(
            name=s["name"],
            description=s["description"],
            example=s.get("examples", ["(none)"])[0],
        )
        for s in skills
    )
    return (
        f"You are a shopping assistant for {card['name']}.\n\n"
        f"{card['description']}\n\n"
        f"## Available actions\n\n{skills_text}\n\n"
        "## Rules you must follow\n\n"
        "1. NEVER invent or guess a product_id. "
        "Always call browse_products first to find the correct product_id before checkout.\n"
        "2. NEVER describe placing an order, completing a purchase, or confirm a result "
        "unless a tool call was actually made and returned a response. "
        "If you have not called a tool, you do not know what happened.\n"
        "3. Once the user confirms they want to proceed with a checkout, "
        "call the tool immediately — do not describe what you are about to do, just do it.\n"
        "4. Summarise tool results in plain English. Do not show raw JSON to the user.\n"
        "5. If a checkout fails or produces unexpected results, report exactly what the tool returned."
    )


_rpc_seq = 0

def a2a_call(store_base: str, intent: dict) -> dict:
    global _rpc_seq
    _rpc_seq += 1
    payload = {
        "jsonrpc": "2.0",
        "id": f"nb-{_rpc_seq}",
        "method": "SendMessage",
        "params": {
            "message": {
                "role": "ROLE_USER",
                "parts": [{"text": json.dumps(intent)}],
                "messageId": f"msg-{_rpc_seq}",
            }
        },
    }
    r = httpx.post(
        f"{store_base}/rpc",
        json=payload,
        headers={"Content-Type": "application/json", "A2A-Version": "1.0"},
        timeout=30,
    )
    r.raise_for_status()
    envelope = r.json()
    if "error" in envelope:
        return {"error": envelope["error"]}
    text = envelope["result"]["message"]["parts"][0]["text"]
    return json.loads(text)


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "call_agent",
            "description": (
                "Send a JSON intent to the agent and receive a response. "
                "Construct the intent object exactly as documented in the available actions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "intent": {
                        "type": "object",
                        "description": 'A JSON object with an "action" field and parameters as documented.',
                    }
                },
                "required": ["intent"],
            },
        },
    }
]


In [ ]:
# ---------------------------------------------------------------------------
# Initialise — run once; re-run to reset the conversation
# ---------------------------------------------------------------------------

card          = fetch_card(GARDEN_STORE)
system_prompt = build_system_prompt(card)
llm           = OpenAI(base_url=f"{OLLAMA_BASE}/v1", api_key="ollama")
messages      = [{"role": "system", "content": system_prompt}]

print(f"Connected to : {card['name']}")
print(f"Model        : {OLLAMA_MODEL}")
print(f"Skills       : {', '.join(s['name'] for s in card.get('skills', []))}")

In [ ]:

# ---------------------------------------------------------------------------
# Chat UI
# ---------------------------------------------------------------------------

CSS = """
<style>
.msg-you   { background:#e8f4e8; border-left:3px solid #3a5c2c; padding:8px 12px; margin:4px 0; border-radius:4px; color:#1a3a10; }
.msg-agent { background:#f0f7ff; border-left:3px solid #2c5c8c; padding:8px 12px; margin:4px 0; border-radius:4px; color:#0d2a4a; }
.msg-tool  { background:#fafafa; border-left:3px solid #aaa;    padding:6px 12px; margin:2px 0; border-radius:4px;
             font-family:monospace; font-size:0.85em; color:#444; }
.msg-error { background:#fff0f0; border-left:3px solid #c0392b; padding:8px 12px; margin:4px 0; border-radius:4px; color:#c0392b; }
</style>
"""

out        = widgets.Output(layout=widgets.Layout(
                border='1px solid #dce8cc',
                border_radius='8px',
                min_height='300px',
                padding='8px',
            ))
text_input = widgets.Text(placeholder='Type your request and press Enter…',
                          layout=widgets.Layout(flex='1'))
send_btn   = widgets.Button(description='Send',       button_style='success')
clear_btn  = widgets.Button(description='Clear chat', button_style='warning')

with out:
    display(widgets.HTML(CSS + f'<p style="color:#555;font-style:italic">Connected to {card["name"]}. Type a request below.</p>'))


def _html(cls, content):
    return widgets.HTML(f'<div class="{cls}">{content}</div>')


def on_send(_):
    user_text = text_input.value.strip()
    if not user_text:
        return

    text_input.value = ''
    send_btn.disabled = True
    send_btn.description = 'Thinking…'

    with out:
        display(_html('msg-you', f'<b>You:</b> {user_text}'))

    messages.append({"role": "user", "content": user_text})

    try:
        while True:
            response = llm.chat.completions.create(
                model=OLLAMA_MODEL,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                with out:
                    display(_html('msg-agent', f'<b>Agent:</b> {msg.content}'))
                break

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                intent = args.get("intent", args)
                result = a2a_call(GARDEN_STORE, intent)
                summary = json.dumps(result)

                with out:
                    display(_html('msg-tool',
                        f'→ {json.dumps(intent)}<br>'
                        f'← {summary[:200] + "…" if len(summary) > 200 else summary}'
                    ))

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                })

    except Exception as e:
        with out:
            display(_html('msg-error', f'<b>Error:</b> {e}'))
    finally:
        send_btn.disabled = False
        send_btn.description = 'Send'


def on_clear(_):
    global messages
    messages = [{"role": "system", "content": system_prompt}]
    out.clear_output()
    with out:
        display(widgets.HTML(CSS + '<p style="color:#555;font-style:italic">Chat cleared.</p>'))


send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)

display(widgets.VBox([
    out,
    widgets.HBox([text_input, send_btn, clear_btn],
                 layout=widgets.Layout(margin='8px 0 0 0', gap='6px'))
]))
